In [1]:
import torch

In [6]:
torch.cuda.is_available()

True

In [2]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 56.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 92.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.4/32.4 MB 84.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 91.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 94.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 95.7 MB/s  0:00:00
  Attempting uninstall: click90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/15 [pyogrio]
    Found existing installation: click 8.2.1━━━━━━━━━━━━━━━━━━  2/15 [pyogrio]
    Uninstalling click-8.2.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/15 [pyogrio]
      Successfully uninstalled click-8.2.1━━━━━━━━━━━━━━━━━━━━  2/15 [pyogrio]
  Attempting uninstall: matplotlib━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/15 [pyogrio]
    Found existing installation: matplotlib 3.10.5━

In [6]:
import logging
from pathlib import Path
import boto3
from botocore.config import Config
from botocore import UNSIGNED
from botocore.exceptions import NoCredentialsError, ClientError

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def download_s3_folder(bucket_name: str, folder_name: str, local_dir: str = "./data") -> None:
    s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
    prefix = folder_name.strip("/")
    if prefix:
        prefix = f"{prefix}/"

    local_path = Path(local_dir)
    local_path.mkdir(parents=True, exist_ok=True)

    try:
        paginator = s3.get_paginator("list_objects_v2")
        found_any = False
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            if "Contents" not in page:
                continue
            found_any = True
            for obj in page["Contents"]:
                key = obj["Key"]
                if key.endswith("/") or key == prefix:
                    continue
                target = local_path / key
                target.parent.mkdir(parents=True, exist_ok=True)
                logger.info(f"Downloading {key} -> {target}")
                s3.download_file(bucket_name, key, str(target))

        if not found_any:
            logger.warning(f"No objects found in '{folder_name}' in bucket '{bucket_name}'")
        else:
            logger.info(f"Downloaded '{folder_name}' from '{bucket_name}' to '{local_dir}'")

    except NoCredentialsError:
        logger.error("AWS credentials not found.")
        raise
    except ClientError as e:
        logger.error(f"AWS client error: {e}")
        raise


# Just call it directly in a notebook cell:
download_s3_folder(
    bucket_name="osapiens-terra-challenge",
    folder_name="makeathon-challenge",
    local_dir="./data",
)

INFO:__main__:Downloading makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2020.tiff -> data/makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2020.tiff
INFO:__main__:Downloading makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2021.tiff -> data/makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2021.tiff
INFO:__main__:Downloading makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2022.tiff -> data/makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2022.tiff
INFO:__main__:Downloading makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2023.tiff -> data/makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2023.tiff
INFO:__main__:Downloading makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2024.tiff -> data/makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2024.tiff
INFO:__main__:Downloading makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2025.tiff -> data/makeathon-challenge/aef-embeddings/test/18NVJ_1_6_2025.tiff
INFO:__main__:Downloading makeathon-challenge/aef-embeddings/test/18NYH_2_1_

In [7]:
import logging
from boto3 import client
from botocore.config import Config
from botocore import UNSIGNED
from botocore.exceptions import ClientError, NoCredentialsError

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


def get_s3_prefix_size(bucket_name: str, folder_name: str = ""):
    s3 = client("s3", config=Config(signature_version=UNSIGNED))

    prefix = folder_name.strip("/")
    if prefix:
        prefix = f"{prefix}/"

    total_bytes = 0
    object_count = 0

    try:
        paginator = s3.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if key.endswith("/") or key == prefix:
                    continue
                total_bytes += obj["Size"]
                object_count += 1

        return total_bytes, object_count

    except NoCredentialsError:
        logger.error("AWS credentials not found.")
        raise
    except ClientError as e:
        logger.error(f"AWS client error: {e}")
        raise


def human_readable_size(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024


total_bytes, count = get_s3_prefix_size(
    bucket_name="osapiens-terra-challenge",
    folder_name="makeathon-challenge",
)

print(f"Objects: {count}")
print(f"Total size: {total_bytes} bytes")
print(f"Readable: {human_readable_size(total_bytes)}")

Objects: 4411
Total size: 45123775280 bytes
Readable: 42.02 GB


In [9]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

def list_s3_keys(bucket_name: str, prefix: str):
    s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
    paginator = s3.get_paginator("list_objects_v2")

    keys = []
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith("/"):
                keys.append(key)

    for k in keys:
        print(k)

    print(f"\nTotal files: {len(keys)}")
    return keys

label_keys = list_s3_keys(
    bucket_name="osapiens-terra-challenge",
    prefix="makeathon-challenge/labels/"
)

makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alert21.tif
makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alert22.tif
makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alert23.tif
makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alert24.tif
makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alert25.tif
makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alertDate21.tif
makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alertDate22.tif
makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alertDate23.tif
makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alertDate24.tif
makeathon-challenge/labels/train/gladl/gladl_18NWG_6_6_alertDate25.tif
makeathon-challenge/labels/train/gladl/gladl_18NWH_1_4_alert21.tif
makeathon-challenge/labels/train/gladl/gladl_18NWH_1_4_alert22.tif
makeathon-challenge/labels/train/gladl/gladl_18NWH_1_4_alert23.tif
makeathon-challenge/labels/train/gladl/gladl_18NWH_1_4_alert24.tif
makeathon-challenge/labels/train/gladl/gla